# SemMap · Исправленный Colab-пайплайн: тексты → ConceptNet → Parquet

**Чистая версия без monkey-patch, под Python 3.12+, T4/L4.**

Данные:
- [Txt.zip](https://drive.google.com/file/d/1I8dYdI4eLVmlhZjNewxl63EGgJKLmDqR/view) — исходные тексты на Google Drive.
- [cstr/conceptnet-normalized-multi](https://huggingface.co/datasets/cstr/conceptnet-normalized-multi) — SQLite ConceptNet 5.5, автоматически скачивается на локальный SSD Colab.
- [Папка результатов](https://drive.google.com/drive/folders/1C7r8A0L0afrfq_8KTebOBM4BJqbZjXM4) — отдельная папка `text-corpus-<EXPERIMENT_LABEL>`.

**Исправлено:** работа с нормализованной SQLite как с колонкой `language`, так и без неё; восстановление после сбоя при существующем `progress.json`; запуск Python-модулей в отдельных процессах после editable-install. **Никаких переопределений методов SQLite, pathlib или модулей в ядре Colab нет.**

Версия учитывает только **английский язык**. Подготовка текстовых графов использует CPU; T4/L4 можно затем задействовать для численных алгоритмов semgraphex. Объём данных ограничивается `MAX_DOCS`, без скрытого отсечения рёбер.

Для повторного запуска оставляйте прежние входные данные и **ту же метку**. Если старый эксперимент создан до исправления исходников, сохранённые графы остаются на Drive, но при несовместимом `run.json` будет предложено начать новую метку.

In [ ]:
# Основные параметры
EXPERIMENT_LABEL = "semmap-text-hf55-spacy-fixed-01"
SOURCE_ZIP_ID = "1I8dYdI4eLVmlhZjNewxl63EGgJKLmDqR"
RESULT_PARENT_ID = "1C7r8A0L0afrfq_8KTebOBM4BJqbZjXM4"
HF_DATASET_REVISION = "4d3fd17e80543e88d9823e2f4fe47901dcf9a4e6"

REPO_BRANCH = "feature/text-corpus-colab-drive-resume"
REPO_DIR = "/content/semgraphex-text-clean"  # отдельная чистая копия кода
WORKSPACE = "/content/semmap-text-corpus"     # вычисления и кэш БД на SSD
MAX_DOCS = 100                                # None: весь ZIP
WORKERS = 0                                   # 0: автоматический подбор
THREADS_PER_WORKER = 1
MEMORY_LIMIT_PER_WORKER = "1GB"
MAX_EDGES_PER_DOC = 20_000_000
SPACY_MODEL = "en_core_web_sm"  # None: без spaCy


In [ ]:
from google.colab import drive, auth
drive.mount("/content/drive")
auth.authenticate_user()

from pathlib import Path
from googleapiclient.discovery import build
import google.auth
import os, sys, subprocess, json, shutil

credentials, _ = google.auth.default()
drive_api = build("drive", "v3", credentials=credentials, cache_discovery=False)
root_id = drive_api.files().get(fileId="root", fields="id").execute()["id"]

def mounted_drive_path(file_id):
    """Find a Drive file via verified parent IDs; no hardcoded MyDrive subpaths."""
    pieces, current = [], file_id
    for _ in range(40):
        if current == root_id:
            return Path("/content/drive/MyDrive").joinpath(*reversed(pieces))
        info = drive_api.files().get(
            fileId=current, fields="id,name,parents"
        ).execute()
        pieces.append(info["name"])
        parents = info.get("parents") or []
        if len(parents) != 1:
            raise RuntimeError(
                f"Файл {file_id} не имеет одного доступного родителя в MyDrive"
            )
        current = parents[0]
    raise RuntimeError("Drive parent chain too deep")

archive_path = mounted_drive_path(SOURCE_ZIP_ID)
result_parent_path = mounted_drive_path(RESULT_PARENT_ID)
if not archive_path.is_file():
    raise FileNotFoundError(f"Исходный ZIP недоступен: {archive_path}")
if not result_parent_path.is_dir():
    raise FileNotFoundError(f"Папка результатов недоступна: {result_parent_path}")
print("Исходный ZIP:", archive_path)
print("Результаты:", result_parent_path / ("text-corpus-" + EXPERIMENT_LABEL))


In [ ]:
import platform
import psutil

cpu_count = os.cpu_count() or 2
available_ram = psutil.virtual_memory().available
free_disk = shutil.disk_usage("/content").free
workers = WORKERS or max(1, min(12, cpu_count - 1, int(available_ram // (2 * 1024**3))))
print("Python:", platform.python_version())
print(f"CPU: {cpu_count}, RAM available: {available_ram / 2**30:.1f} GiB")
print(f"Local disk free: {free_disk / 2**30:.1f} GiB, workers: {workers}")
if sys.version_info < (3, 12):
    raise RuntimeError("Нужен Colab Python 3.12+. Измените runtime.")
if free_disk < 4_000_000_000:
    raise RuntimeError("Недостаточно SSD для SQLite (~1.78 GB), ZIP и результатов.")
try:
    import torch
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("GPU не назначен (построение графов выполняется на CPU)")
except ImportError:
    print("GPU диагностика недоступна; построение графов всё равно CPU")


In [ ]:
# Клонируем исходники в ОТДЕЛЬНЫЙ рабочий каталог без старых патчей.
repo = Path(REPO_DIR)
if not (repo / ".git").is_dir():
    if repo.exists():
        raise FileExistsError(f"{repo} существует, но не является Git-репозиторием")
    subprocess.run([
        "git", "clone", "--single-branch", "--branch", REPO_BRANCH,
        "https://github.com/SemanticMap/semgraphex.git", str(repo)
    ], check=True)
else:
    subprocess.run(["git", "fetch", "origin", REPO_BRANCH], cwd=repo, check=True)
    subprocess.run(["git", "checkout", REPO_BRANCH], cwd=repo, check=True)
    subprocess.run(["git", "reset", "--hard", "origin/" + REPO_BRANCH], cwd=repo, check=True)

extras = "[text,text-lemma]" if SPACY_MODEL else "[text]"
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-e", str(repo) + extras
], check=True)

revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
print("Revision:", revision)
# Тестируем импорт в НОВОМ процессе: editable src-layout будет корректно подключён.
subprocess.run([
    sys.executable, "-c",
    "import semmap_haken.text_colab as m; "
    "print('Import OK:', m.__file__)"
], check=True, cwd=repo)


In [ ]:
# Автоматическая установка и проверка spaCy (никаких monkey-patch).
if SPACY_MODEL:
    if SPACY_MODEL != "en_core_web_sm":
        raise ValueError("В текущем англоязычном режиме используйте en_core_web_sm")
    check_model = (
        "import spacy; p=spacy.load('en_core_web_sm'); "
        "assert p('Cats are running.')[0].lemma_.lower() == 'cat'; "
        "print('spaCy + en_core_web_sm OK')"
    )
    first_check = subprocess.run([sys.executable, "-c", check_model], cwd=repo)
    if first_check.returncode:
        subprocess.run([
            sys.executable, "-m", "spacy", "download", SPACY_MODEL
        ], check=True, cwd=repo)
        subprocess.run([sys.executable, "-c", check_model], check=True, cwd=repo)
else:
    print("spaCy выключена: остаётся точное сопоставление слов.")


In [ ]:
# Скачиваем фиксированную ревизию HF SQLite на локальный SSD.
# Схема проверяется БЕЗ динамического изменения функций.
hub_code = """
from semmap_haken.text_hf import download_database
import json, sys
info = download_database(
    cache_dir=sys.argv[1], revision=sys.argv[2]
)
print(json.dumps(info, ensure_ascii=False, indent=2))
"""
hub_cache = str(Path(WORKSPACE) / "hf-cache")
subprocess.run([
    sys.executable, "-c", hub_code, hub_cache, HF_DATASET_REVISION
], check=True, cwd=repo)


## Создание графов и сохранение контрольных точек

На каждом завершённом документе обновляются Parquet-файлы и `progress.json` в папке эксперимента на Google Drive. При **Run all** с той же меткой восстановятся результаты, для которых присутствуют `run.json` и проверенные контрольные суммы.

Если в прежнем эксперименте только `experiment.json`, `progress.json` и пустая папка `documents`, запуск начнётся заново без ошибки. Если есть готовые графы, но отсутствует `run.json`, выполнение остановится, **не удаляя** файлы: такие данные нельзя безопасно принять за проверенную контрольную точку.

Текстовый этап CPU-bound: выбор T4/L4 влияет только на дальнейшие вычисления.

In [ ]:
cmd = [
    sys.executable, "-u", "-m", "semmap_haken.text_colab",
    "--archive", str(archive_path),
    "--hf-db",
    "--hf-revision", HF_DATASET_REVISION,
    "--drive-parent", str(result_parent_path),
    "--label", EXPERIMENT_LABEL,
    "--workspace", WORKSPACE,
    "--workers", str(workers),
    "--threads-per-worker", str(THREADS_PER_WORKER),
    "--memory-limit", MEMORY_LIMIT_PER_WORKER,
    "--max-edges-per-document", str(MAX_EDGES_PER_DOC),
]
if MAX_DOCS is not None:
    cmd += ["--max-docs", str(MAX_DOCS)]
if SPACY_MODEL:
    cmd += ["--spacy-model", SPACY_MODEL]
print("Запуск:", " ".join(cmd), flush=True)
# Чистый дочерний интерпретатор Python импортирует код с диска,
# поэтому никакие патчи ядра Colab не нужны и не используются.
subprocess.run(cmd, check=True, cwd=repo)


In [ ]:
result_dir = result_parent_path / ("text-corpus-" + EXPERIMENT_LABEL)
print("Папка эксперимента:", result_dir)
for name in ("progress.json", "manifest.json"):
    path = result_dir / name
    if path.is_file():
        data = json.loads(path.read_text(encoding="utf-8"))
        if name == "progress.json":
            print("Прогресс:", json.dumps(data, ensure_ascii=False, indent=2))
        else:
            print("Число документов:", len(data.get("documents", [])))
            print("Всего слов:", data.get("word_count"))
            print("Всего рёбер:", data.get("edge_count"))
print("COMPLETED:", (result_dir / "COMPLETED").is_file())


### При прерывании Colab

Перезапустите сеанс и выполните **Run all**, оставив прежнюю `EXPERIMENT_LABEL`, исходный ZIP, ревизию ConceptNet и модель spaCy. Скачивание Hugging Face повторится при новом VM, но завершённые документы будут восстановлены с Google Drive.

При смене версии исходного кода, модели или данных `run.json` может не совпасть — в этом случае используйте новую метку. **Не удаляйте старые папки** и не вставляйте патчи `Path.iterdir`, `sqlite3.connect` или `NormalizedConceptNet.__init__`.